In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [23]:
# Recommended: import the stable helper from the Python module
from data_utils import get_clean_X_y
X, y = get_clean_X_y()
print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (5644, 98)
y shape: (5644,)


In [24]:
# Create a train/test split for modeling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4515, 98), (1129, 98), (4515,), (1129,))

In [25]:
# Train a Gaussian Naive Bayes classifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
gnb = GaussianNB()
gnb.fit(X_train, y_train)
print('Training accuracy:', gnb.score(X_train, y_train))

Training accuracy: 0.9991140642303433


In [26]:
# Cross-validation on the training set (5-fold) to estimate generalization
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(GaussianNB(), X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print('CV scores (5-fold) on training set:', cv_scores)
print('CV mean accuracy:', cv_scores.mean(), 'std:', cv_scores.std())

CV scores (5-fold) on training set: [0.99889258 0.99778516 1.         0.99889258 1.        ]
CV mean accuracy: 0.9991140642303433 std: 0.0008287170291858253


In [27]:
# Evaluate on the test set
y_pred = gnb.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print('')
print('Classification report:\n', classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

Test accuracy: 0.9964570416297609

Classification report:
               precision    recall  f1-score   support

           e       1.00      0.99      1.00       693
           p       0.99      1.00      1.00       436

    accuracy                           1.00      1129
   macro avg       1.00      1.00      1.00      1129
weighted avg       1.00      1.00      1.00      1129

Confusion matrix:
 [[689   4]
 [  0 436]]


In [28]:
# Grid search over var_smoothing and retrain the model with the best parameter
from sklearn.model_selection import GridSearchCV
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
grid = GridSearchCV(GaussianNB(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
best_vs = grid.best_params_['var_smoothing']
print('Best var_smoothing:', best_vs)
print('Best CV score:', grid.best_score_)
# Retrain GaussianNB with the best var_smoothing and evaluate
gnb = GaussianNB(var_smoothing=best_vs)
gnb.fit(X_train, y_train)
print('Training accuracy (best var_smoothing):', gnb.score(X_train, y_train))
# Optionally, cross-validate the best model on full data
cv_scores_best = cross_val_score(GaussianNB(var_smoothing=best_vs), X, y, cv=5, scoring='accuracy', n_jobs=-1)
print('CV (5-fold) on full data with best var_smoothing mean:', cv_scores_best.mean(), 'std:', cv_scores_best.std())

Best var_smoothing: 1e-09
Best CV score: 0.9991140642303433
Training accuracy (best var_smoothing): 0.9991140642303433
CV (5-fold) on full data with best var_smoothing mean: 0.9971653192117547 std: 0.0018063766907864117
